# AutoShield AI: Supplier Intelligence & Alternative Sourcing Engine

## Project Overview

Predicting supply chain disruptions is only the first step toward building resilient automotive supply chains.

To support strategic decision making, disruption probabilities must be converted into actionable supplier intelligence.

This notebook transforms transaction-level risk predictions into supplier-level insights that can be used for:

- Supplier Monitoring
- Alternative Sourcing
- Procurement Prioritization
- Scenario Simulation
- Executive Decision Support

The resulting supplier intelligence layer serves as the decision engine powering the AutoShield AI platform.

---

## Objectives

This notebook aims to:

1. Aggregate transaction-level risk scores.
2. Build supplier intelligence profiles.
3. Create a procurement readiness score.
4. Rank suppliers based on risk and business importance.
5. Generate alternative sourcing recommendations.
6. Support disruption scenario analysis.

---

## Expected Outputs

- Supplier Intelligence Dataset
- Procurement Readiness Score
- Supplier Rankings
- Alternative Supplier Recommendations
- Scenario Simulation Inputs

# 1. Import Required Libraries

The following libraries are used for data aggregation, supplier intelligence generation, and procurement analytics.

In [1]:
import pandas as pd
import numpy as np

# 2. Load Risk Scored Dataset

The dataset generated by the Supplier Risk Prediction Agent contains disruption probability scores for every supply chain transaction.

These probabilities will now be aggregated into supplier-level intelligence metrics.

In [2]:
df = pd.read_csv(
    "../data/risk_scored_supply_chain.csv"
)

print(df.shape)

df.head()

(180519, 13)


,Order Id,Supplier,Commodity,Order Country,Order Region,Shipping Mode,Days for shipping (real),Days for shipment (scheduled),Delay_Days,Late_delivery_risk,Order Item Quantity,Sales,Risk_Probability
0,77202,Indonesia,Battery Materials,Indonesia,Southeast Asia,Standard Class,3,4,-1,0,1,327.75,0.000019
1,75939,India,Battery Materials,India,South Asia,Standard Class,5,4,1,1,1,327.75,0.962055
2,75938,India,Battery Materials,India,South Asia,Standard Class,4,4,0,0,1,327.75,0.000018
3,75937,Australia,Battery Materials,Australia,Oceania,Standard Class,3,4,-1,0,1,327.75,0.000013
4,75936,Australia,Battery Materials,Australia,Oceania,Standard Class,2,4,-2,0,1,327.75,0.000013


# 3. Supplier-Commodity Intelligence Layer

Supply chain decisions are rarely made at the supplier level alone.

A supplier may be highly reliable for one commodity while posing elevated risk for another.

To support commodity-specific sourcing decisions, intelligence profiles are created at the Supplier-Commodity level.

Each profile contains:

- Supplier
- Commodity
- Order Volume
- Revenue Contribution
- Average Risk Score

This structure enables more granular procurement and disruption analysis.

In [3]:
supplier_directory = (
    df.groupby(
        [
            "Supplier",
            "Commodity"
        ]
    )
    .agg(
        orders=("Order Id","count"),
        sales=("Sales","sum"),
        avg_risk=("Risk_Probability","mean")
    )
    .reset_index()
)

supplier_directory.head()

,Supplier,Commodity,orders,sales,avg_risk
0,AfganistÃ¡n,Battery Materials,17,4731.000134,0.668447
1,AfganistÃ¡n,Interior Components,31,4610.510118,0.609848
2,AfganistÃ¡n,Mechanical Assemblies,4,1276.650031,0.719891
3,AfganistÃ¡n,Metal Components,40,10782.920174,0.475930
4,AfganistÃ¡n,Plastic Components,19,9124.349989,0.601601


# Business Interpretation

Suppliers with:

- High disruption probability
- High delivery delays
- Significant revenue exposure

represent critical supply chain risks and require closer monitoring.

Conversely, suppliers with low risk and high operational capacity may serve as strong alternative sourcing candidates.

# 4. Procurement Readiness Score

A procurement score is developed to identify suppliers that are both reliable and strategically important.

The score combines:

- Supplier Risk
- Revenue Contribution
- Order Volume

This approach prioritizes suppliers capable of supporting large-scale procurement requirements while maintaining acceptable risk levels.

In [4]:
supplier_directory["sales_norm"] = (
    supplier_directory["sales"]
    /
    supplier_directory["sales"].max()
)

supplier_directory["orders_norm"] = (
    supplier_directory["orders"]
    /
    supplier_directory["orders"].max()
)

supplier_directory["Procurement_Score"] = (
    (1 - supplier_directory["avg_risk"]) * 0.5
    +
    supplier_directory["sales_norm"] * 0.3
    +
    supplier_directory["orders_norm"] * 0.2
)

In [5]:
supplier_directory.head()

,Supplier,Commodity,orders,sales,avg_risk,sales_norm,orders_norm,Procurement_Score
0,AfganistÃ¡n,Battery Materials,17,4731.000134,0.668447,0.003079,0.001903,0.167081
1,AfganistÃ¡n,Interior Components,31,4610.510118,0.609848,0.003001,0.003470,0.196670
2,AfganistÃ¡n,Mechanical Assemblies,4,1276.650031,0.719891,0.000831,0.000448,0.140393
3,AfganistÃ¡n,Metal Components,40,10782.920174,0.475930,0.007018,0.004477,0.265036
4,AfganistÃ¡n,Plastic Components,19,9124.349989,0.601601,0.005938,0.002127,0.201406


# Procurement Score Interpretation

Higher scores indicate suppliers that:

- Exhibit lower disruption risk
- Handle larger procurement volumes
- Contribute significant revenue

These suppliers are valuable candidates for strategic sourcing and supply chain resilience initiatives.

# 5. Supplier Ranking

Suppliers are ranked using the Procurement Readiness Score.

The resulting ranking supports:

- Procurement Prioritization
- Strategic Supplier Selection
- Alternative Sourcing Decisions

In [6]:
supplier_ranking = (
    supplier_directory
    .sort_values(
        "Procurement_Score",
        ascending=False
    )
)

supplier_ranking.head(20)

,Supplier,Commodity,orders,sales,avg_risk,sales_norm,orders_norm,Procurement_Score
261,Estados Unidos,Semiconductors,8934,1.358367e+06,0.554026,0.884031,1.000000,0.688196
259,Estados Unidos,Metal Components,4382,1.536561e+06,0.543385,1.000000,0.490486,0.626404
257,Estados Unidos,Interior Components,5542,8.646063e+05,0.557581,0.562689,0.620327,0.514082
270,EtiopÃ­a,Metal Components,3,1.099930e+03,0.000016,0.000716,0.000336,0.500274
126,Burkina Faso,Semiconductors,6,7.399700e+02,0.000015,0.000482,0.000672,0.500271
234,Emiratos Ãrabes Unidos,Battery Materials,3,7.559300e+02,0.000015,0.000492,0.000336,0.500207
599,OmÃ¡n,Semiconductors,4,5.199600e+02,0.000015,0.000338,0.000448,0.500183
271,EtiopÃ­a,Plastic Components,4,4.498200e+02,0.000017,0.000293,0.000448,0.500169
486,Macedonia,Semiconductors,3,5.500000e+02,0.000016,0.000358,0.000336,0.500167
309,Grecia,Battery Materials,2,5.999600e+02,0.000016,0.000390,0.000224,0.500154


In [7]:
supplier_ranking.tail(20)

,Supplier,Commodity,orders,sales,avg_risk,sales_norm,orders_norm,Procurement_Score
823,TÃºnez,Interior Components,4,399.930012,0.969193,0.000260,0.000448,0.015571
763,SudÃ¡n del Sur,Semiconductors,3,529.980011,0.969589,0.000345,0.000336,0.015376
293,GabÃ³n,Mechanical Assemblies,1,119.970001,0.969904,0.000078,0.000112,0.015094
241,Eritrea,Semiconductors,1,129.990005,0.969997,0.000085,0.000112,0.015049
272,EtiopÃ­a,Semiconductors,5,629.990006,0.970388,0.000410,0.000560,0.015041
694,Ruanda,Mechanical Assemblies,1,89.970001,0.970191,0.000059,0.000112,0.014945
650,Qatar,Mechanical Assemblies,1,59.980000,0.972889,0.000039,0.000112,0.013590
243,Eslovaquia,Interior Components,1,79.980003,0.973725,0.000052,0.000112,0.013175
402,Israel,Mechanical Assemblies,4,419.839992,0.974151,0.000273,0.000448,0.013096
848,UzbekistÃ¡n,Mechanical Assemblies,4,285.000000,0.975537,0.000185,0.000448,0.012377


# 6. Risk Categorization

To simplify executive decision making, suppliers are grouped into risk categories.

Risk segmentation allows supply chain managers to quickly identify suppliers requiring immediate attention.

In [8]:
supplier_directory["Risk_Level"] = pd.cut(
    supplier_directory["avg_risk"],
    bins=[0,0.4,0.6,1],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [9]:
supplier_directory[
    [
        "Supplier",
        "avg_risk",
        "Risk_Level"
    ]
].head()

,Supplier,avg_risk,Risk_Level
0,AfganistÃ¡n,0.668447,High
1,AfganistÃ¡n,0.609848,High
2,AfganistÃ¡n,0.719891,High
3,AfganistÃ¡n,0.475930,Medium
4,AfganistÃ¡n,0.601601,High


# 7. Alternative Sourcing Framework

A core objective of resilient supply chains is maintaining operational continuity during supplier disruptions.

The alternative sourcing framework identifies suppliers with:

- Similar sourcing capabilities
- Lower disruption risk
- Strong procurement readiness

These recommendations are later used by the Scenario Simulation Engine.

In [10]:
def get_alternatives(
    supplier_country,
    commodity,
    supplier_directory
):

    alternatives = supplier_directory[
        supplier_directory["Commodity"] == commodity
    ].copy()

    alternatives = alternatives[
        alternatives["Supplier"] != supplier_country
    ]

    alternatives = alternatives.sort_values(
        "Procurement_Score",
        ascending=False
    )

    return alternatives

# Example Alternative Sourcing Scenario

The following example demonstrates how the system identifies replacement suppliers for a disrupted sourcing location.

In [11]:
get_alternatives(
    "China",
    "Semiconductors",
    supplier_directory
).head()

,Supplier,Commodity,orders,sales,avg_risk,sales_norm,orders_norm,Procurement_Score,Risk_Level
261,Estados Unidos,Semiconductors,8934,1.358367e+06,0.554026,0.884031,1.000000,0.688196,Medium
126,Burkina Faso,Semiconductors,6,7.399700e+02,0.000015,0.000482,0.000672,0.500271,Low
599,OmÃ¡n,Semiconductors,4,5.199600e+02,0.000015,0.000338,0.000448,0.500183,Low
486,Macedonia,Semiconductors,3,5.500000e+02,0.000016,0.000358,0.000336,0.500167,Low
779,Surinam,Semiconductors,3,3.799800e+02,0.000015,0.000247,0.000336,0.500134,Low


# 8. Scenario Simulation Support

The supplier intelligence layer supports disruption simulations by estimating the business impact of supplier failures.

For a selected supplier and commodity combination, the system can:

- Estimate revenue exposure
- Identify affected procurement volume
- Recommend alternative sourcing options
- Support contingency planning

This capability forms the basis of the Scenario Simulation Engine used within AutoShield AI.

In [12]:
def simulate_disruption(
    country,
    commodity,
    supplier_directory
):

    affected = supplier_directory[
        (supplier_directory["Supplier"] == country)
        &
        (supplier_directory["Commodity"] == commodity)
    ]

    impacted_sales = (
        affected["sales"].sum()
    )

    alternatives = get_alternatives(
        country,
        commodity,
        supplier_directory
    )

    return {
        "country": country,
        "commodity": commodity,
        "impacted_sales": impacted_sales,
        "alternatives": alternatives
    }

# Example scenario

In [13]:
simulation = simulate_disruption(
    "China",
    "Semiconductors",
    supplier_directory
)

print(
    simulation["impacted_sales"]
)

simulation["alternatives"]

288134.71662944


,Supplier,Commodity,orders,sales,avg_risk,sales_norm,orders_norm,Procurement_Score,Risk_Level
261,Estados Unidos,Semiconductors,8934,1.358367e+06,0.554026,0.884031,1.000000,0.688196,Medium
126,Burkina Faso,Semiconductors,6,7.399700e+02,0.000015,0.000482,0.000672,0.500271,Low
599,OmÃ¡n,Semiconductors,4,5.199600e+02,0.000015,0.000338,0.000448,0.500183,Low
486,Macedonia,Semiconductors,3,5.500000e+02,0.000016,0.000358,0.000336,0.500167,Low
779,Surinam,Semiconductors,3,3.799800e+02,0.000015,0.000247,0.000336,0.500134,Low
...,...,...,...,...,...,...,...,...,...
686,RepÃºblica de Gambia,Semiconductors,1,2.000000e+02,0.962708,0.000130,0.000112,0.018707,High
336,Guinea Ecuatorial,Semiconductors,1,1.999900e+02,0.966932,0.000130,0.000112,0.016595,High
763,SudÃ¡n del Sur,Semiconductors,3,5.299800e+02,0.969589,0.000345,0.000336,0.015376,High
241,Eritrea,Semiconductors,1,1.299900e+02,0.969997,0.000085,0.000112,0.015049,High


# 9. Export Supplier Intelligence Dataset

The final supplier intelligence dataset is exported for use in:

- War Room Dashboard
- Supplier Risk Explorer
- Alternative Sourcing Engine
- Scenario Simulator
- Executive Copilot

In [14]:
supplier_directory.to_csv(
    "../data/final_supplier_directory.csv",
    index=False
)

# Conclusion

This notebook transformed disruption probabilities into actionable supplier directory.

Key outcomes include:

- Supplier Risk Aggregation
- Procurement Readiness Scoring
- Supplier Ranking
- Risk Categorization
- Alternative Sourcing Support
- Scenario Simulation Support

The resulting directory layer enables automotive organizations to move beyond disruption detection and toward proactive supply chain resilience planning.

In [15]:
supplier_directory.columns.tolist()

['Supplier',
 'Commodity',
 'orders',
 'sales',
 'avg_risk',
 'sales_norm',
 'orders_norm',
 'Procurement_Score',
 'Risk_Level']

In [16]:
df["Commodity"].unique()

array(['Battery Materials', 'Interior Components', 'Semiconductors',
       'Metal Components', 'Mechanical Assemblies', 'Plastic Components'],
      dtype=object)